# Attention-LSTM Training
This notebook trains a score-prediction model from over-by-over engineered data.

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Masking, LSTM, Dense, Attention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

PROCESSED_FILE = 'processed_data.csv'
MODEL_FILE = 'ball_outcome_model.keras'

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


In [8]:
def create_3d_sequences(processed_df, max_overs=20, num_features=6):
    import numpy as np

    print("Creating expanding window sequences to prevent data leakage...")

    feature_cols = [
        "runs_this_over",
        "wickets_this_over",
        "cumulative_runs",
        "cumulative_wickets",
        "run_rate",
        "balls_remaining",
    ]

    # Optional safety: keep chronological order if an over column exists
    if "over" in processed_df.columns:
        processed_df = processed_df.sort_values(["match_id", "over"], kind="stable")

    grouped = list(processed_df.groupby("match_id", sort=False))
    total_samples = sum(max(len(g) - 1, 0) for _, g in grouped)

    X = np.zeros((total_samples, max_overs, num_features), dtype=np.float32)
    y = np.empty((total_samples, 1), dtype=np.float32)

    pos = 0
    for _, match_data in grouped:
        features = match_data[feature_cols].to_numpy(dtype=np.float32, copy=False)
        overs_played = features.shape[0]
        if overs_played <= 1:
            continue

        if overs_played > max_overs + 1:
            # Keep behavior safe for longer innings
            features = features[: max_overs + 1]
            overs_played = features.shape[0]

        final_score = float(match_data["final_score"].iloc[0])
        n = overs_played - 1  # number of snapshots

        # Build all expanding snapshots for this match
        match_X = np.zeros((n, max_overs, num_features), dtype=np.float32)

        # Vectorized fill:
        # row j is present in snapshots j..n-1 at timestep j
        for j in range(n):
            match_X[j:, j, :] = features[j]

        X[pos : pos + n] = match_X
        y[pos : pos + n, 0] = final_score
        pos += n

    return X, y

def build_model(time_steps=20, num_features=6):
    inputs = Input(shape=(time_steps, num_features))

    # Ignore 0.0 padded values
    masked_inputs = Masking(mask_value=0.0)(inputs)

    # LSTM output for every timestep
    lstm_out = LSTM(64, return_sequences=True)(masked_inputs)

    # Self-attention
    attention_out = Attention()([lstm_out, lstm_out])

    # Sequence compression
    pooled_out = GlobalAveragePooling1D()(attention_out)

    # Regression head
    x = Dense(32, activation='relu')(pooled_out)
    outputs = Dense(1, activation='linear')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

In [9]:
processed_df = pd.read_csv(PROCESSED_FILE)
print('Processed shape:', processed_df.shape)

X, y = create_3d_sequences(processed_df)

# Train/validation split (80/20)
split_idx = int(0.8 * len(X))
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print('Training data shape:', X_train.shape)
print('Validation data shape:', X_val.shape)

Processed shape: (95025, 9)
Creating expanding window sequences to prevent data leakage...
Training data shape: (72089, 20, 6)
Validation data shape: (18023, 20, 6)


In [10]:
model = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 20, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 20, 6)     │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_1 (Masking) │ (None, 20, 6)     │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, 20)        │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 20, 64)    │     18,176 │ masking_1[0][0],  │
│                     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 20, 64)    │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convert_to_tensor_1 │ (None, 20)        │          0 │ any_1[0][0]       │
│ (ConvertToTensor)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention_1[0][0… │
│ (GlobalAveragePool… │                   │            │ convert_to_tenso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,289 (79.25 KB)

 Trainable params: 20,289 (79.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
print('Starting Training (Targeting MAE < 10)...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

Starting Training (Targeting MAE < 10)...
Epoch 1/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 27.1849 - mae: 27.1849 - val_loss: 18.1021 - val_mae: 18.1021
Epoch 2/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 18.3743 - mae: 18.3743 - val_loss: 18.1710 - val_mae: 18.1710
Epoch 3/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 18.2380 - mae: 18.2380 - val_loss: 17.8574 - val_mae: 17.8574
Epoch 4/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 18.1556 - mae: 18.1556 - val_loss: 17.9411 - val_mae: 17.9411
Epoch 5/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 18.1129 - mae: 18.1129 - val_loss: 18.1125 - val_mae: 18.1125
Epoch 6/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - loss: 18.0487 - mae: 18.0487 - val_loss: 17.9305 - val_mae: 17.9305
Epoch 7/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 18.0496 - mae: 18.0496 - val_loss: 18.4058 - val_mae: 18.4058
Epoch 8/30
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 18.0135 - mae: 18.01

In [12]:
val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
print(f'Validation MAE: {val_mae:.4f}')

model.save(MODEL_FILE)
print(f'Saved trained model to {MODEL_FILE}')

Validation MAE: 18.1340
Saved trained model to ball_outcome_model.keras
